# Modèle Numérique de Terrain (MNT) à partir du nuage de points Lidar

Ce notebook vérifie que `processing.lidar.rasterize.compute_mnt` extrait les points sol (classification LAS = 2, déjà attribuée par l'IGN dans les dalles Lidar HD) et les rasterise (PDAL `writers.gdal`) en un raster d'altitude sol, sur le nuage fusionné produit par le notebook 15.

Le raster produit (traité) est écrit dans `data/processed/raster/lidar/`.

In [ ]:
import sys
from pathlib import Path

# GeoDataEngine/ (racine du package core/) et son parent (racine du repo, contient data/)
project_root = Path.cwd().parent
repo_root = project_root.parent
sys.path.insert(0, str(project_root))

import numpy as np
import rasterio

from processing.lidar.rasterize import compute_mnt

merged_path = repo_root / "data" / "raw" / "raster" / "lidar" / "nuage_points.las"
output_path = repo_root / "data" / "processed" / "raster" / "lidar" / "mnt.tif"

## 1. Génération du MNT

In [ ]:
result = compute_mnt(merged_path, output_path, resolution=0.5)
print("MNT ecrit :", result)
print(f"Taille : {result.stat().st_size / 1_000_000:.2f} Mo")

## 2. Vérification : CRS, emprise, résolution, altitudes

In [ ]:
with rasterio.open(result) as src:
    print("CRS :", src.crs)
    print("Resolution :", src.res)
    print("Taille (pixels) :", src.width, "x", src.height)
    print("Bounds :", src.bounds)
    print("NoData :", src.nodata)
    mnt = src.read(1)

valid = mnt[mnt != -9999]
print(f"\nPixels valides : {valid.size} / {mnt.size} ({100 * valid.size / mnt.size:.1f}%)")
print(f"Altitude min/max/moyenne : {valid.min():.2f} / {valid.max():.2f} / {valid.mean():.2f} m")

## 3. Aperçu du MNT

In [ ]:
import matplotlib.pyplot as plt

masked = np.ma.masked_equal(mnt, -9999)

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(masked, cmap="terrain", extent=[src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top])
ax.set_title("MNT (altitude sol, m) - dalle Lidar HD")
fig.colorbar(im, ax=ax, label="Altitude (m)")
plt.show()